In [78]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params
from macromolecules.macromolecule import Macromolecule
from expression.build_me_model import flatten_list

In [137]:
from utils.parameters import human_model as m_model
mu_val = 1e-9
n_cores = 10
base = 0
counter = 5

lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme_new = pickle.load(handle)

def add_sink(m, tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type ='sink')
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    if not os.path.isfile('test_sinks2.tab'):
        with open('test_sinks2.tab', 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open('test_sinks2.tab', 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')
        

def _add_sink(metabs_, tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    if type(metabs_) != list:
        metabs_ = list(metabs_)

    for m in metabs_:
        if isinstance(m, cobra.Metabolite): # object
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type ='sink')
        else: # string
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type ='sink')
    
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat

In [41]:
tme0 = tme_new
sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)

tme1, sln1, stat1 = _add_sink(metabs_ = [m.id for m in m_model.metabolites])

/home/hratch/Projects/human_me/scripts/core/model.py:306 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 27.1794 seconds with status 0


In [49]:
# tolerance = max([abs(v) for v in tme1.infeasible_reactions(mu_val = mu_val, sln = sln1, stat = stat1, tolerance = 0).values()])

# res = pd.DataFrame(columns = ['flux'])
# for r_id, flux in tme0.infeasible_reactions(mu_val = mu_val, sln = sln0, stat = stat0, tolerance = tolerance).items():
#     res.loc[r_id, 'flux'] = flux
# res['abs_flux'] = res.flux.abs()
# res.sort_values(by = 'abs_flux', ascending = False, inplace = True)
# test_metabs = list()
# for r_id in res.head(10).index:
#     test_metabs += [m.id for m in list(tme0.reactions.get_by_id(r_id).metabolites) if not hasattr(m, 'type')]
# test_metabs = sorted(set([r_id for r_id in test_metabs if 'biomass' not in r_id]))
# tme2, sln2, stat2 = _add_sink(metabs_ = test_metabs) # 0

# from itertools import product
# test = sorted(set(flatten_list([[m.id for m in tme0.reactions.get_by_id(r_id).metabolites if not isinstance(m, Macromolecule)] for r_id in flux_res.head(10).index])))
# test += [i[0] + i[1] for i in list(product(['a', 'c', 'u', 't'], ['mp_c', 'tp_c', 'dp_c']))]



In [55]:
tme_gdp, sln_gdp, stat_gdp = _add_sink(metabs_ = ['gdp_c'])
flux_res = pd.DataFrame(index = [r.id for r in tme0.reactions], columns = ['flux0', 'flux_gdp'])
for i in flux_res.index:
    flux_res.loc[i,:] = [sln0[tme0.reactions.index(i)], sln_gdp[tme_gdp.reactions.index(i)]]
flux_res['gdp - 0'] = flux_res.flux_gdp - flux_res.flux0
flux_res['abs_difference'] = flux_res['gdp - 0'].abs()
flux_res.sort_values(by = 'abs_difference', ascending = False, inplace = True)
flux_res['metabolites'] = pd.Series(flux_res.index).apply(lambda x: {m.id: c for m,c in tme0.reactions.get_by_id(x).metabolites.items()}).tolist()
flux_res['reaction'] = pd.Series(flux_res.index).apply(lambda x: tme0.reactions.get_by_id(x).reaction).tolist()
# flux_res = flux_res[flux_res.abs_difference > 1e-25]
flux_res.to_csv('for_erick.csv')

Getting MINOS parameters...
Done in 64.5991 seconds with status 0


In [134]:
from itertools import product
test = sorted(set(flatten_list([[m.id for m in tme0.reactions.get_by_id(r_id).metabolites if not isinstance(m, Macromolecule)] for r_id in flux_res.head(10).index])))
test += [i[0] + i[1] for i in list(product(['a', 'c', 'u', 'g'], ['mp_c', 'tp_c', 'dp_c']))]




In [142]:
res = pd.read_csv('test_sinks2.tab', sep = '\t').sort_values(by = ['status', 'metabolite_id']).reset_index(drop = True)

In [108]:
test_[test_.metabolite_id.isin(test)].shape

(19, 2)

In [109]:
len(test)

31

In [67]:
pd.Series(flux_res.index).apply(lambda x: {m.id: c for m,c in tme0.reactions.get_by_id(x).metabolites.items()})

0       {'h2o_c': 1.0, 'h2o_m': -1.0, 'HGNC:642_folded...
1       {'HGNC:10989_folded_pre_protein_m': -0.0001147...
2       {'h_i': -4.0, 'ATPS4m_1_complex_i': -2.2080074...
3       {'HGNC:10991_folded_pre_protein_m': -0.0001328...
4                           {'ach_e': -1.0, 'ach_b': 1.0}
                              ...                        
8672    {'HGNC:16889_unprocessed_folded_protein_r': 1,...
8673    {'HGNC:16889_folded_protein_r': 1, 'ala_L_r': ...
8674    {'HGNC:17350_lariat_n': -1, 'h2o_n': -70275, '...
8675    {'HGNC:17350_mrna_c': -1, 'HGNC:17350_mrna_deg...
8676    {'h2o_m': -1.0, 'b2coa_m': -1.0, '3hbcoa_R_m':...
Length: 8677, dtype: object

In [17]:
test = ['m_ids'

import multiprocessing
import gc

pool = multiprocessing.Pool(processes = n_cores)
try:
    res = pool.map(add_sink, test)
    pool.close()
    pool.join()
    gc.collect()
except:
    pool.close()
    pool.join()
    gc.collect()
    raise ValueError('Parallelization failed')

Getting MINOS parameters...
Done in 57.9268 seconds with status 1
